### Overview

In this notebook, we will select the area of interest (AOI) for our fire dashboard. We will use [DuckDB](https://duckdb.org/) to query a global admin boundaries dataset directly from the cloud and save one Admin2 region (district) as a GeoJSON file.

### Overview of the Task

We will query the Admin2 boundaries provided by [FieldMaps](https://fieldmaps.io/data/), pick the district of Sangrur in Punjab, India and save it as `aoi.geojson` in the `data/` folder. The dashboard will display this file as the AOI outline.

### Setup

Determine our runtime environment.

In [ ]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'

print(f'Environment: {environment}')

If we are on Google Colab, install the required packages. Local runtimes are expected to have the packages already installed.

In [ ]:
%%capture
if environment in ['colab', 'colab_enterprise']:
  !pip install duckdb

Import packages.

In [ ]:
import duckdb
import geopandas as gpd
import os

Create the data folder. Both data files used by the dashboard are saved here.

In [ ]:
data_folder = 'data'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)

### Query the Admin2 Boundaries

We initialize DuckDB and enable the spatial extension.

In [ ]:
con = duckdb.connect()
con.install_extension('spatial')
con.load_extension('spatial')

The source is a 2 GB GeoParquet file with Admin2 boundaries from [geoBoundaries](https://www.geoboundaries.org/) for the whole world. Instead of downloading it, we query it with [read_parquet()](https://duckdb.org/docs/stable/data/parquet/overview) and fetch only the rows and columns we need.

In [ ]:
parquet_url = 'https://data.fieldmaps.io/edge-matched/open/intl/adm2_polygons.parquet'

First, we list all Admin1 names (States/Provinces) in our country. The `adm0_src` column uses the [3-digit ISO code](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-3) for each country. Filtering on the country first keeps the query fast.

In [ ]:
country = 'IND'

query = f'''
SELECT DISTINCT adm1_name
FROM read_parquet('{parquet_url}')
WHERE adm0_src = '{country}'
ORDER BY adm1_name
'''

admin1_df = con.sql(query).df()
admin1_df

Next, we fetch all Admin2 polygons in our chosen Admin1 region. We convert the geometry to WKB with `ST_AsWKB()` so GeoPandas can read it.

In [ ]:
adm1_name = 'Punjab'

query = f'''
SELECT adm0_name, adm1_name, adm2_name, adm2_id, ST_AsWKB(geometry) AS geometry
FROM read_parquet('{parquet_url}')
WHERE
  adm0_src = '{country}' AND
  adm1_name = '{adm1_name}'
'''

admin2_df = con.sql(query).df()
admin2_df

We turn the results into a GeoPandas GeoDataFrame by specifying the geometry column and the CRS.

In [ ]:
admin2_gdf = gpd.GeoDataFrame(
    admin2_df,
    geometry=gpd.GeoSeries.from_wkb(admin2_df['geometry'].apply(bytes)),
    crs='EPSG:4326')

We can visualize the Admin2 polygons with [explore()](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html). Hover over a polygon to see its name.

In [ ]:
admin2_gdf.explore(tooltip='adm2_name')

### Select the AOI

We select one Admin2 region by name. Sangrur is a good choice because it has a high number of fires from stubble burning in October and November. Replace the name with any `adm2_name` from the table above.

In [ ]:
adm2_name = 'Sangrur'

aoi_gdf = admin2_gdf[admin2_gdf['adm2_name'] == adm2_name]
aoi_gdf

In [ ]:
aoi_gdf.explore()

### Save the Results

We save the AOI as a GeoJSON file with [to_file()](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.to_file.html).

> Note: On Google Colab, files saved to the local filesystem are deleted when the runtime is disconnected. Download `aoi.geojson` and place it in the `data/` folder of your dashboard.

In [ ]:
output_path = os.path.join(data_folder, 'aoi.geojson')
aoi_gdf.to_file(output_path)
print(f'Saved to {output_path}')